# Prediction

We split years into train (first 80%) and test (last 20%), predict median GPU price from BTC, S&P 500 return, and—when available—supply chain returns (TSMC, NVIDIA, AMD, Micron). We compare test RMSE to a naive rule: next year = this year's price. With few years, the naive rule often wins; supply chain features can still improve the model.

## Train/test split

Chronological: earlier years for training, latest for testing.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_PROC = ROOT / "data" / "processed"

In [2]:
df = pd.read_csv(DATA_PROC / "merged_yearly.csv")
feat_base = ["btc_avg_price", "sp500_return"]
feat_sc = [c for c in ["tsmc_return", "nvda_return", "amd_return", "micron_return"] if c in df.columns]
feat = feat_base + feat_sc
df = df.dropna(subset=["median_gpu_price"] + feat).sort_values("year").reset_index(drop=True)
n = len(df)
split = int(0.8 * n)
train, test = df.iloc[:split], df.iloc[split:]
print(f"Train: {train['year'].min()}-{train['year'].max()} ({len(train)} yrs)")
print(f"Test:  {test['year'].min()}-{test['year'].max()} ({len(test)} yrs)")
print(f"Features: {feat}")

Train: 2014.0-2022.0 (8 yrs)
Test:  2023.0-2025.0 (3 yrs)
Features: ['btc_avg_price', 'sp500_return', 'tsmc_return', 'qcom_return']


## Fit and test RMSE

In [3]:
X_tr = train[feat]
y_tr = train["median_gpu_price"]
model = LinearRegression().fit(X_tr, y_tr)

X_te = test[feat]
y_te = test["median_gpu_price"]
pred = model.predict(X_te)
rmse = np.sqrt(mean_squared_error(y_te, pred))
print(f"Regression test RMSE: {rmse:.2f}")

Regression test RMSE: 6544.58


## Baseline: previous year's price

In [4]:
baseline_pred = test["median_gpu_price"].shift(1)
baseline_pred.iloc[0] = train["median_gpu_price"].iloc[-1]
rmse_baseline = np.sqrt(mean_squared_error(y_te, baseline_pred))
print(f"Naive baseline test RMSE: {rmse_baseline:.2f}")
print(f"Regression beats baseline: {rmse < rmse_baseline}")

Naive baseline test RMSE: 1334.77
Regression beats baseline: False


## Interpretation

- **Small test set:** With only a few years in test, RMSE is volatile; the naive baseline (previous year's price) is a strong benchmark.
- **When regression wins:** If supply chain and macro features (BTC, SP500, TSMC/NVDA/AMD/Micron) are in the model and the test period aligns with demand shifts, the model can beat the baseline.
- **When baseline wins:** Year-over-year median GPU price is often sticky or mean-reverting; then "next year = this year" is hard to beat with few predictors and few observations.